## Neighbourhood simulation
Where does a sound-driven signal actually go in the real wiring, when the path is not picked in advance?

---

### Init

In [ ]:
import pandas as pd
from pathlib import Path
import pyarrow.dataset as ds
import pyarrow.compute as pc
import duckdb
import networkx as nx

root = Path.cwd().parent
# root = Path(__file__).resolve().parent.parent

neurons_file = root / 'data' / 'body-annotations-male-cns-v1.0-minconf-0.5.feather'
connections_file = root / 'data' / 'connectome-weights-male-cns-v1.0-minconf-0.5.feather'

df = pd.read_feather(neurons_file) # directly to pandas dataframe
traced = df[df['status'] == 'Traced']
id_to_superclass = traced.set_index('bodyId')['superclass'].to_dict()

# lazy handle on the big file, nothing is loaded until a filtered read asks for it
dataset = ds.dataset(connections_file, format="feather")


# common functions and lookups copied from the earlier sections of 05

# get outgoing connections of specified neurons
def query_edges(neuron_ids, min_weight=3):
    if not isinstance(neuron_ids, (list, set, tuple)):
        neuron_ids = [neuron_ids]
    expr = pc.field("body_pre").isin(neuron_ids) & (pc.field("weight") >= min_weight)
    return dataset.to_table(filter=expr).to_pandas()

def grow_neighbourhood(start_ids, n_hops, min_weight=3):
    if not isinstance(start_ids, (list, set, tuple)):
        start_ids = [start_ids]

    seen = set(start_ids)       # every neuron reached so far, starting ones included
    to_check = set(start_ids)   # neurons whose outgoing connections we look up this hop
    all_edges = []

    for hop in range(1, n_hops + 1):
        hop_edges = query_edges(to_check, min_weight=min_weight)
        all_edges.append(hop_edges)

        to_check = set(hop_edges['body_post']) - seen   # targets not seen before
        # seen |= to_check
        seen.update(to_check)

        # prints each hop statistics
        print(f"hop {hop}: {len(hop_edges)} edges, {len(to_check)} new neurons, {len(seen)} total")
        if not to_check:
            break

    return pd.concat(all_edges, ignore_index=True), seen

motor_info = (traced[traced['superclass'] == 'vnc_motor']
              .set_index('bodyId')[['somaNeuromere', 'instance']]
              .fillna({'somaNeuromere': 'unknown'}) # added missing values
              .to_dict('index'))

# regrow neighbourhood again
jo_neurons = traced[traced['type'].str.contains('JO', na=False)]

jo_auditory = jo_neurons[jo_neurons['type'].str.contains('JO-A|JO-B', na=False, regex=True)]

jo_ids = set(jo_auditory['bodyId'])

edges_jo, seen_jo = grow_neighbourhood(jo_ids, n_hops=4, min_weight=10)

leg_segments = ['T1', 'T2', 'T3']

abdomen_segments = [f'A{i}' for i in range(1, 11)]

##### Changing the number of hops

**goal:** check whether the leg versus abdomen gap is just a side effect of stopping at 4 hops. neurons at the last hop have no outgoing connections in the table, and leg motor neurons sit further from the brain, so they might be under-counted. this section grows the neighbourhood one more hop and compares how much incoming wiring each group has.

so the baseline from the old experiments were:
- cutoff 10, 138 sources with input 10,000, 4 hops, decay 0.8, 80 ticks
- 154,662 neurons in the neighbourhood
- the cliff at 361, and at 361 about 34,819 neurons fired with legs 75% and abdomen 91%
- prediction: with 5 hops, if the cutoff is the cause, the leg distribution should move towards the abdominal one (a higher 25th percentile, fewer with no incoming edges). if the legs stay low, the lead is in the wiring.
- the leg and abdomen motor-neuron totals from this table (total incoming weight per motor neuron):

| group | neurons | median | 25th percentile | max | no incoming edges | ceiling >= 361 |
|---|---|---|---|---|---|---|
| abdomen | 207 | 2,137 | 1,257 | 9,565 | 4 / 207 = 1.9% | 187 / 207 = 90.3% |
| leg | 500 | 1,235 | 163 | 16,748 | 32 / 500 = 6.4% | 319 / 500 = 63.8% |

In [35]:
edges_jo5, seen_jo5 = grow_neighbourhood(jo_ids, n_hops=5, min_weight=10)

hop 1: 1159 edges, 335 new neurons, 473 total
hop 2: 20534 edges, 7413 new neurons, 7886 total
hop 3: 345251 edges, 57210 new neurons, 65096 total
hop 4: 1412733 edges, 89566 new neurons, 154662 total
hop 5: 844426 edges, 20188 new neurons, 174850 total


now check, did the leg motor neurons gained incoming wiring?

In [36]:
# uses motor_info, leg_segments and abdomen_segments
def motor_totals(edges):
    incoming_total = edges.groupby('body_post')['weight'].sum()
    motor_in = pd.DataFrame(
        [(nid, info['somaNeuromere'], incoming_total.get(nid, 0)) for nid, info in motor_info.items()],
        columns=['bodyId', 'segment', 'incoming'])
    motor_in['group'] = motor_in['segment'].map(
        lambda s: 'leg' if s in leg_segments else ('abdomen' if s in abdomen_segments else 'unknown'))
    return motor_in

for label, edges in [("4 hops", edges_jo), ("5 hops", edges_jo5)]:
    m = motor_totals(edges)
    print(label)
    for group, g in m.groupby('group'):
        n_none = (g['incoming'] == 0).sum()
        print(f"  {group}: {len(g)} neurons, no incoming {n_none} = {n_none / len(g) * 100:.1f}%")
        print(g['incoming'].describe(percentiles=[.25, .5, .75]).round(0).to_string())

4 hops
  abdomen: 207 neurons, no incoming 4 = 1.9%
count     207.0
mean     2632.0
std      2058.0
min         0.0
25%      1258.0
50%      2137.0
75%      3487.0
max      9565.0
  leg: 500 neurons, no incoming 32 = 6.4%
count      500.0
mean      2876.0
std       3554.0
min          0.0
25%        164.0
50%       1236.0
75%       4813.0
max      16748.0
  unknown: 1 neurons, no incoming 0 = 0.0%
count      1.0
mean     719.0
std        NaN
min      719.0
25%      719.0
50%      719.0
75%      719.0
max      719.0
5 hops
  abdomen: 207 neurons, no incoming 1 = 0.5%
count      207.0
mean      3046.0
std       2331.0
min          0.0
25%       1532.0
50%       2502.0
75%       4070.0
max      10904.0
  leg: 500 neurons, no incoming 30 = 6.0%
count      500.0
mean      3207.0
std       3932.0
min          0.0
25%        176.0
50%       1486.0
75%       5486.0
max      18460.0
  unknown: 1 neurons, no incoming 0 = 0.0%
count       1.0
mean     1138.0
std         NaN
min      1138.0
25%   

> both groups gained about the same proportion at the median (20% for legs, 17% for abdomen)
>
> if less hops had hurt legs specifically, they should have gained much more
>
> so the 4-hop truncation does not explain most of the gap.

now check does the simulation result survive a bigger neighbourhood?